# IndoBERT — balanced 3000 (Colab GPU) — *tanpa clone repo*

Jalur ANDAL tanpa GitHub PAT. `train_indobert.py` self-contained, jadi cukup **upload**
2 file dari repo lokalmu:
- `src/modeling/train_indobert.py`
- `outputs/labeling/balanced_3000.csv`

**Runtime → Change runtime type → GPU (T4)** sebelum mulai.

## 1. Dependensi

In [ ]:
%pip install -q 'transformers>=4.40' torch 'pymongo>=4' dnspython certifi scikit-learn matplotlib pandas numpy python-dotenv

## 2. Upload 2 file (train_indobert.py + balanced_3000.csv)

In [ ]:
from google.colab import files
print('Pilih KEDUA file: train_indobert.py DAN balanced_3000.csv')
up = files.upload()
assert 'train_indobert.py' in up and 'balanced_3000.csv' in up, 'Kedua file harus terupload.'
print('OK:', list(up))

## 3. Set MONGO_URI + jalankan training
`processed_bert` dibaca dari Mongo Atlas (butuh IP allowlist `0.0.0.0/0`). Subset di-filter
ke 3000 baris balanced → split kanonik 2100/600/300 (identik SVM).

In [ ]:
import os
from getpass import getpass
os.environ['MONGO_URI'] = os.environ.get('MONGO_URI') or getpass('MONGO_URI: ')

# Standalone (bukan -m): file self-contained. --reports-dir . -> artefak di /content.
!python train_indobert.py --subset balanced_3000.csv --tag balanced3k --reports-dir .

## 4. Lihat hasil + unduh artefak

In [ ]:
import json
from IPython.display import Image, display
m = json.load(open('indobert_balanced3k_metrics.json'))
print(json.dumps(m, ensure_ascii=False, indent=2))
display(Image('indobert_balanced3k_test_confusion.png'))

In [ ]:
from google.colab import files
files.download('indobert_balanced3k_metrics.json')
files.download('indobert_balanced3k_test_confusion.png')

## 5. Setelah unduh
Taruh `indobert_balanced3k_metrics.json` di `outputs/reports/` lokal, lalu:
```bash
python -m src.modeling.compare_models --tag balanced3k
```